# Цель: Проверить, можно ли использовать party_public_id как разметку, и создать обучающие пары

# Валидация

In [1]:
import pandas as pd
import numpy as np
import sys

import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append('..')
from src.evaluation.validation import (
    get_public_id_stats,
    estimate_cluster_similarity,
    mark_trusted_public_ids,
)

from src.features.pair_generation import (
    generate_positive_pairs,
    generate_easy_negative_pairs,
    generate_hard_negative_pairs,
    attach_pair_columns,
)

In [2]:
# ============================================================
# LOAD PROCESSED DATASET
# ============================================================

df = pd.read_parquet(
    "../data/processed/processed_dataset.parquet"
)

print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")

df.head()

Rows: 3,653,581
Columns: 17


,record_id,country,source_snapshot_date,relation_kind,relation_role,party_type,company_public_id,company_name_norm,party_public_id,party_name,ownership_share_pct,script,snapshot_date,name_normalized,name_no_legal,name_latin,name_length
0,arm_rec_49df8429c4e1912e923f09210d5d2c39,arm,2023-01-09,shareholder,beneficial_owner,individual,arm_company_e8217b23afa28741,արոմատոս,NaN,Անդրանիկ Աբարյան,100.0,armenian,2023-01-09,անդրանիկ աբարյան,անդրանիկ աբարյան,andranik abaryan,16
1,arm_rec_ee7466e19e5683c426051f8f94e13930,arm,2024-02-06,shareholder,beneficial_owner,individual,arm_company_997ee60051c4fda1,տրապիզոն քոնսթրաքշն,NaN,Տիգրան Հարությունյան,50.0,armenian,2024-02-06,տիգրան հարությունյան,տիգրան հարությունյան,tigran haroutyounyan,20
2,arm_rec_344e2597b9b2d65611c3ea6689aeb786,arm,2022-03-02,shareholder,beneficial_owner,individual,arm_company_ba593702ed7d5006,անհայտ կազմակերպություն,NaN,Նարեկ Նարգիզյան,30.0,armenian,2022-03-02,նարեկ նարգիզյան,նարեկ նարգիզյան,narek nargizyan,15
3,arm_rec_bb45f92e3f36dcbedeb82569d875a6f9,arm,2024-01-18,shareholder,beneficial_owner,individual,arm_company_dbdf69c5afa462f3,լեոն ա յակուբյան քոնթրաքթինգ,NaN,ՎԱՍԿԵՆ ՅԱԿՈՒԲՅԱՆ,100.0,armenian,2024-01-18,վասկեն յակուբյան,վասկեն յակուբյան,vasken yakoubyan,16
4,arm_rec_d4ffbf80b5b31e075113302d085b674f,arm,2024-01-30,shareholder,beneficial_owner,individual,arm_company_f39e74b9a9df4d74,սանտաֆամիլիա,NaN,Ժիրայր Ավանյան,15.0,armenian,2024-01-30,ժիրայր ավանյան,ժիրայր ավանյան,zhirayr avanyan,14


In [3]:
# ============================================================
# PARTY_PUBLIC_ID COVERAGE
# ============================================================

labeled_mask = df["party_public_id"].notna()

print("=" * 60)
print("PARTY_PUBLIC_ID COVERAGE")
print("=" * 60)

print(f"Rows with party_public_id: {labeled_mask.sum():,}")
print(f"Rows without party_public_id: {(~labeled_mask).sum():,}")
print(f"Coverage: {labeled_mask.mean() * 100:.2f}%")

print("\nUnique party_public_id:")
print(df["party_public_id"].nunique())

PARTY_PUBLIC_ID COVERAGE
Rows with party_public_id: 112,141
Rows without party_public_id: 3,541,440
Coverage: 3.07%

Unique party_public_id:
53427


In [4]:
# ============================================================
# PUBLIC ID BASIC STATS
# ============================================================

public_id_stats = get_public_id_stats(df)

print("=" * 60)
print("PUBLIC ID BASIC STATS")
print("=" * 60)

display(public_id_stats.head())

print("\nCluster size stats:")
display(public_id_stats["cluster_size"].describe())

print("\nUnique names per ID stats:")
display(public_id_stats["unique_names"].describe())

PUBLIC ID BASIC STATS


,party_public_id,cluster_size,unique_names,countries,scripts,unique_name_ratio
0,arm_party_082bb6f73073b2c7,40,1,1,1,0.025000
1,arm_party_08c10e0c2e663101,13783,2,1,1,0.000145
2,arm_party_0f4ae5e4da91f1ec,20,1,1,1,0.050000
3,arm_party_159246b7ea94e907,80,1,1,1,0.012500
4,arm_party_19c10b8409fdcc52,125,1,1,1,0.008000



Cluster size stats:


count    53427.000000
mean         2.098957
std         61.638288
min          1.000000
25%          1.000000
50%          1.000000
75%          2.000000
max      13783.000000
Name: cluster_size, dtype: float64


Unique names per ID stats:


count    53427.000000
mean         1.000168
std          0.012978
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max          2.000000
Name: unique_names, dtype: float64

In [5]:
# ============================================================
# LARGEST PUBLIC ID CLUSTERS
# ============================================================

largest_clusters = (
    public_id_stats
    .sort_values("cluster_size", ascending=False)
    .head(20)
)

display(largest_clusters)

,party_public_id,cluster_size,unique_names,countries,scripts,unique_name_ratio
1,arm_party_08c10e0c2e663101,13783,2,1,1,0.000145
41,arm_party_e37a0726e4bad216,2595,2,1,1,0.000771
19,arm_party_5e037f5ac852c3bf,1422,1,1,1,0.000703
35,arm_party_b9e7bd6a8eda69ac,1372,1,1,1,0.000729
28,arm_party_a8aa1e01dc7e1534,1180,1,1,1,0.000847
17,arm_party_54db7bcf8f23be5d,520,2,1,1,0.003846
34,arm_party_b25dce88d0b4c1b2,408,1,1,1,0.002451
27,arm_party_a22eeccd27959921,336,2,1,1,0.005952
31,arm_party_abea8f0a66567aca,316,2,1,1,0.006329
26,arm_party_9e0901689ff90734,240,2,1,1,0.008333


In [6]:
# ============================================================
# EXAMPLES OF LARGE CLUSTERS
# ============================================================

large_ids = (
    public_id_stats
    .sort_values("cluster_size", ascending=False)
    .head(5)["party_public_id"]
    .tolist()
)

for public_id in large_ids:
    print("=" * 80)
    print(f"party_public_id: {public_id}")

    subset = df[df["party_public_id"] == public_id]

    print(f"Rows: {len(subset):,}")
    print(f"Unique names: {subset['name_latin'].nunique():,}")

    display(
        subset[
            [
                "party_name",
                "name_latin",
                "country",
                "relation_kind",
                "relation_role",
            ]
        ]
        .drop_duplicates()
        .head(30)
    )

party_public_id: arm_party_08c10e0c2e663101
Rows: 13,783
Unique names: 2


,party_name,name_latin,country,relation_kind,relation_role
3701,ՊՐՈՄԻՇԼԵՆՆԱՅԱ ԿՈՄՊԱՆԻԱ,promishlennaya kompania,arm,shareholder,beneficial_owner
3717,ԱՐԴՅՈՒՆԱԲԵՐԱԿԱՆ ԸՆԿԵՐՈՒԹՅՈՒՆ ԲԸ,ardyounaberakan ynkeroutyoun by,arm,shareholder,beneficial_owner
8624,Պրոմիշլեննայա կոմպանիա,promishlennaya kompania,arm,shareholder,beneficial_owner


party_public_id: arm_party_e37a0726e4bad216
Rows: 2,595
Unique names: 2


,party_name,name_latin,country,relation_kind,relation_role
3720,«ՍԹԱՐ ԴԱՍԹ»,star dast,arm,shareholder,beneficial_owner
3887,«ՍԹԱՐ ԴԱՍԹ» ՓԲԸ,star dast pby,arm,shareholder,beneficial_owner


party_public_id: arm_party_5e037f5ac852c3bf
Rows: 1,422
Unique names: 1


,party_name,name_latin,country,relation_kind,relation_role
3886,ՈՒՐԲԱՆԵՎԵՆՏ ՊԼՅՈՒՍ ՍՊԸ,ourbanevent plyous spy,arm,shareholder,beneficial_owner
14052,Ուրբանեվենտ Պլյուս ՍՊԸ,ourbanevent plyous spy,arm,shareholder,beneficial_owner


party_public_id: arm_party_b9e7bd6a8eda69ac
Rows: 1,372
Unique names: 1


,party_name,name_latin,country,relation_kind,relation_role
3716,«ԱՄՓ ՀՈԼԴԻՆԳՍ» ՓԲԸ,amp holdings pby,arm,shareholder,beneficial_owner


party_public_id: arm_party_a8aa1e01dc7e1534
Rows: 1,180
Unique names: 1


,party_name,name_latin,country,relation_kind,relation_role
3690,Ստարտ Աէրո ՍՊԸ,start aero spy,arm,shareholder,beneficial_owner


In [7]:
# ============================================================
# ESTIMATE NAME SIMILARITY INSIDE PUBLIC ID CLUSTERS
# ============================================================

similarity_stats = estimate_cluster_similarity(df)

public_id_stats = public_id_stats.merge(
    similarity_stats,
    on="party_public_id",
    how="left"
)

display(public_id_stats.head())

print("\nAverage similarity stats:")
display(public_id_stats["avg_name_similarity"].describe())

,party_public_id,cluster_size,unique_names,countries,scripts,unique_name_ratio,avg_name_similarity,min_name_similarity
0,arm_party_082bb6f73073b2c7,40,1,1,1,0.025000,100.000000,100.000000
1,arm_party_08c10e0c2e663101,13783,2,1,1,0.000145,25.925926,25.925926
2,arm_party_0f4ae5e4da91f1ec,20,1,1,1,0.050000,100.000000,100.000000
3,arm_party_159246b7ea94e907,80,1,1,1,0.012500,100.000000,100.000000
4,arm_party_19c10b8409fdcc52,125,1,1,1,0.008000,100.000000,100.000000



Average similarity stats:


count    53427.000000
mean        99.996704
std          0.361955
min         25.925926
25%        100.000000
50%        100.000000
75%        100.000000
max        100.000000
Name: avg_name_similarity, dtype: float64

In [8]:
# ============================================================
# CROSS-LANGUAGE CLUSTERS
# ============================================================

cross_language = public_id_stats[
    public_id_stats["avg_name_similarity"] < 50
]

print(f"Cross-language clusters: {len(cross_language):,}")

display(
    cross_language.head(20)
)

Cross-language clusters: 1


,party_public_id,cluster_size,unique_names,countries,scripts,unique_name_ratio,avg_name_similarity,min_name_similarity
1,arm_party_08c10e0c2e663101,13783,2,1,1,0.000145,25.925926,25.925926


In [9]:
public_id_stats = mark_trusted_public_ids(
    public_id_stats,
    max_cluster_size=20_000,
    max_unique_names=3,
    max_countries=3,
    max_scripts=3,
)

print("=" * 60)
print("TRUSTED PUBLIC ID ANALYSIS")
print("=" * 60)

print(public_id_stats["is_trusted"].value_counts())

trusted_ids = set(
    public_id_stats[
        public_id_stats["is_trusted"]
    ]["party_public_id"]
)

noisy_ids = set(
    public_id_stats[
        ~public_id_stats["is_trusted"]
    ]["party_public_id"]
)

print(f"\nTrusted IDs: {len(trusted_ids):,}")
print(f"Noisy IDs: {len(noisy_ids):,}")

TRUSTED PUBLIC ID ANALYSIS
is_trusted
True    53427
Name: count, dtype: int64

Trusted IDs: 53,427
Noisy IDs: 0


# Вывод после валидации

**Покрытие party_public_id**

Поле party_public_id присутствует примерно у 3% записей датасета:

- 112 141 строк имеют party_public_id;
- всего найдено 53 427 уникальных идентификаторов.

Несмотря на относительно небольшое покрытие, этого объема достаточно для построения обучающей выборки и оценки качества моделей Entity Resolution.

**Качество party_public_id**

Анализ показал, что party_public_id обладает очень высокой консистентностью:

- среднее количество уникальных имен внутри одного party_public_id практически равно 1;
- большинство кластеров содержат одно и то же имя;
- практически отсутствуют случаи смешения разных сущностей внутри одного идентификатора.
  
Это позволяет использовать party_public_id как  частично размеченные данные для обучения и оценки модели.

**Размеры кластеров**

Большинство party_public_id имеют размер 1–2 записи.

При этом были обнаружены крупные кластеры:

- 13 783 записей;
- 2 595 записей;
- 1 422 записей и т.д.

Однако анализ показал, что это не ошибки разметки, а повторяющиеся упоминания одной и той же сущности в разных строках датасета.

**Multilingual / cross-language случаи**

Был найден только один кластер с очень низкой строковой похожестью имен:

promishlennaya kompania

и

ardyounaberakan ynkeroutyoun by

При этом оба названия относятся к одной сущности и представляют собой разные языковые варианты записи.

Это показывает, что:

- низкая строковая похожесть не всегда означает разные сущности;
- обычные string similarity методы недостаточны для multilingual Entity Resolution;
- системе потребуется поддержка:
  - транслитерации;
  - multilingual embeddings;
  - контекстных признаков.

**Итог validation**

В результате проверки все 53 427 party_public_id были признаны пригодными для использования в качестве обучающей разметки.

Таким образом:

- party_public_id можно использовать для генерации positive pairs;
- разные party_public_id можно использовать для negative pairs;
- датасет пригоден для supervised matching pipeline.

# Генерация обучающих пар

In [10]:
positive_pairs = generate_positive_pairs(
    df=df,
    trusted_ids=trusted_ids,
    max_pairs_per_id=50,
)

print("=" * 60)
print("POSITIVE PAIRS")
print("=" * 60)

print(f"Positive pairs: {len(positive_pairs):,}")
display(positive_pairs.head())

POSITIVE PAIRS
Positive pairs: 50,338


,idx1,idx2,label,pair_type
0,297881,397530,1,positive_same_public_id
1,34821,39314,1,positive_same_public_id
2,4022,302874,1,positive_same_public_id
3,339655,397530,1,positive_same_public_id
4,50823,418932,1,positive_same_public_id


In [11]:
easy_negative_pairs = generate_easy_negative_pairs(
    df=df,
    trusted_ids=trusted_ids,
    n_pairs=min(len(positive_pairs), 100_000),
)

print("=" * 60)
print("EASY NEGATIVE PAIRS")
print("=" * 60)

print(f"Easy negative pairs: {len(easy_negative_pairs):,}")
display(easy_negative_pairs.head())

EASY NEGATIVE PAIRS
Easy negative pairs: 50,338


,idx1,idx2,label,pair_type
0,3704916,282140,0,easy_negative_different_public_id
1,56377,3718302,0,easy_negative_different_public_id
2,3657154,3653204,0,easy_negative_different_public_id
3,3650362,350523,0,easy_negative_different_public_id
4,3717636,261877,0,easy_negative_different_public_id


In [12]:
hard_negative_pairs = generate_hard_negative_pairs(
    df=df,
    trusted_ids=trusted_ids,
    sample_size=50_000,
    threshold=80,
    max_pairs=min(len(positive_pairs), 100_000),
)

print("=" * 60)
print("HARD NEGATIVE PAIRS")
print("=" * 60)

print(f"Hard negative pairs: {len(hard_negative_pairs):,}")
display(hard_negative_pairs.head())

HARD NEGATIVE PAIRS
Hard negative pairs: 50,338


,idx1,idx2,label,pair_type,name_similarity
0,3690485,3702223,0,hard_negative_similar_name,80.645161
1,3701843,3676262,0,hard_negative_similar_name,82.352941
2,3701843,3696115,0,hard_negative_similar_name,80.000000
3,3701843,3671926,0,hard_negative_similar_name,80.645161
4,3701843,3657223,0,hard_negative_similar_name,83.582090


In [13]:
pairs = pd.concat(
    [
        positive_pairs,
        easy_negative_pairs,
        hard_negative_pairs,
    ],
    ignore_index=True,
)

pairs = pairs.sample(
    frac=1,
    random_state=42,
).reset_index(drop=True)

print("=" * 60)
print("LABELED PAIRS")
print("=" * 60)

print("Label distribution:")
print(pairs["label"].value_counts())

print("\nPair types:")
print(pairs["pair_type"].value_counts())

display(pairs.head())

LABELED PAIRS
Label distribution:
label
0    100676
1     50338
Name: count, dtype: int64

Pair types:
pair_type
easy_negative_different_public_id    50338
hard_negative_similar_name           50338
positive_same_public_id              50338
Name: count, dtype: int64


,idx1,idx2,label,pair_type,name_similarity
0,3670911,3701401,0,easy_negative_different_public_id,NaN
1,334475,244796,0,hard_negative_similar_name,90.322581
2,181156,445560,0,hard_negative_similar_name,90.322581
3,414456,244792,0,hard_negative_similar_name,90.322581
4,335053,171471,0,hard_negative_similar_name,90.322581


In [14]:
columns_to_attach = [
    "record_id",
    "party_public_id",
    "party_name",
    "name_latin",
    "name_no_legal",
    "country",
    "script",
    "relation_kind",
    "relation_role",
    "company_public_id",
    "company_name_norm",
]

pairs_full = attach_pair_columns(
    pairs=pairs,
    df=df,
    columns=columns_to_attach,
)

print(f"Pairs shape: {pairs_full.shape}")
display(pairs_full.head())

Pairs shape: (151014, 27)


,idx1,idx2,label,pair_type,name_similarity,record_id_1,party_public_id_1,party_name_1,name_latin_1,name_no_legal_1,...,party_public_id_2,party_name_2,name_latin_2,name_no_legal_2,country_2,script_2,relation_kind_2,relation_role_2,company_public_id_2,company_name_norm_2
0,3670911,3701401,0,easy_negative_different_public_id,NaN,tjk_rec_6b4e838005267448452f5c1133aa4c4d,tjk_party_b045fbfadecdf163,Катабеков Нурмурод Шодимуродович,katabekov nurmurod shodimurodovich,катабеков нурмурод шодимуродович,...,tjk_party_f016271677ad78d6,Нуманов Фахриддин Манонович,numanov fahriddin manonovich,нуманов фахриддин манонович,tjk,cyrillic,director,director,tjk_company_b5e0bd84c92681ff,муассисаи таҳсилоти миёнаи умумии No7
1,334475,244796,0,hard_negative_similar_name,90.322581,arm_rec_38b312f23f109b4ed6adbb069d2b866a,arm_party_ecf2ba27d1eca9fc,«ԱՄՓ ՀՈԼԴԻՆԳ» ՍՊԸ,amp holding spy,ամփ հոլդինգ սպը,...,arm_party_b9e7bd6a8eda69ac,«ԱՄՓ ՀՈԼԴԻՆԳՍ» ՓԲԸ,amp holdings pby,ամփ հոլդինգս փբը,arm,armenian,shareholder,beneficial_owner,arm_company_d2dba87b64a78c27,զանգեզուրի պղնձամոլիբդենային կոմբինատ
2,181156,445560,0,hard_negative_similar_name,90.322581,arm_rec_334e40b0e3226a3cb9c4e73b7cb2cd59,arm_party_ecf2ba27d1eca9fc,«ԱՄՓ ՀՈԼԴԻՆԳ» ՍՊԸ,amp holding spy,ամփ հոլդինգ սպը,...,arm_party_b9e7bd6a8eda69ac,«ԱՄՓ ՀՈԼԴԻՆԳՍ» ՓԲԸ,amp holdings pby,ամփ հոլդինգս փբը,arm,armenian,shareholder,beneficial_owner,arm_company_d2dba87b64a78c27,զանգեզուրի պղնձամոլիբդենային կոմբինատ
3,414456,244792,0,hard_negative_similar_name,90.322581,arm_rec_e2dc59d0be787068f6aabbcd86a22df0,arm_party_b9e7bd6a8eda69ac,«ԱՄՓ ՀՈԼԴԻՆԳՍ» ՓԲԸ,amp holdings pby,ամփ հոլդինգս փբը,...,arm_party_ecf2ba27d1eca9fc,«ԱՄՓ ՀՈԼԴԻՆԳ» ՍՊԸ,amp holding spy,ամփ հոլդինգ սպը,arm,armenian,shareholder,beneficial_owner,arm_company_081b4ce85680db74,զանգեզուրի պղնձամոլիբդենային կոմբինատ փբը
4,335053,171471,0,hard_negative_similar_name,90.322581,arm_rec_42a1de0414cde58f8791666c966cdb5f,arm_party_ecf2ba27d1eca9fc,«ԱՄՓ ՀՈԼԴԻՆԳ» ՍՊԸ,amp holding spy,ամփ հոլդինգ սպը,...,arm_party_b9e7bd6a8eda69ac,«ԱՄՓ ՀՈԼԴԻՆԳՍ» ՓԲԸ,amp holdings pby,ամփ հոլդինգս փբը,arm,armenian,shareholder,beneficial_owner,arm_company_d2dba87b64a78c27,զանգեզուրի պղնձամոլիբդենային կոմբինատ


In [15]:
print("=" * 60)
print("SANITY CHECK")
print("=" * 60)

positive_check = pairs_full[pairs_full["label"] == 1]

same_id_ratio = (
    positive_check["party_public_id_1"]
    ==
    positive_check["party_public_id_2"]
).mean()

print(f"Positive pairs with same public_id: {same_id_ratio:.4%}")

negative_check = pairs_full[pairs_full["label"] == 0]

different_id_ratio = (
    negative_check["party_public_id_1"]
    !=
    negative_check["party_public_id_2"]
).mean()

print(f"Negative pairs with different public_id: {different_id_ratio:.4%}")

SANITY CHECK
Positive pairs with same public_id: 100.0000%
Negative pairs with different public_id: 100.0000%


In [16]:
display(
    pairs_full[pairs_full["label"] == 1][
        [
            "party_name_1",
            "party_name_2",
            "name_latin_1",
            "name_latin_2",
            "pair_type",
        ]
    ]
    .sample(20, random_state=42)
)

,party_name_1,party_name_2,name_latin_1,name_latin_2,pair_type
26570,Хамрокулов Амон Фатоевич,Хамрокулов Амон Фатоевич,hamrokulov amon fatoevich,hamrokulov amon fatoevich,positive_same_public_id
41500,Саидкамолов Мухтор Бурҳонович,Саидкамолов Мухтор Бурҳонович,saidkamolov muhtor burҳonovich,saidkamolov muhtor burҳonovich,positive_same_public_id
83063,Джураев Аминджон Абдуллоджонович,Джураев Аминджон Абдуллоджонович,dzhuraev amindzhon abdullodzhonovich,dzhuraev amindzhon abdullodzhonovich,positive_same_public_id
54379,Ахмедов Убайдуло Ғафурович,Ахмедов Убайдуло Ғафурович,ahmedov ubai dulo gafurovich,ahmedov ubai dulo gafurovich,positive_same_public_id
146191,Ҳафизов Абдуҷабор Сафарович,Ҳафизов Абдуҷабор Сафарович,ҳafizov abduҷabor safarovich,ҳafizov abduҷabor safarovich,positive_same_public_id
64816,Тавакалов Бахтиёр,Тавакалов Бахтиёр,tavakalov bahtie r,tavakalov bahtie r,positive_same_public_id
56030,Тилаков Султонмурод Бобокаримович,Тилаков Султонмурод Бобокаримович,tilakov sultonmurod bobokarimovich,tilakov sultonmurod bobokarimovich,positive_same_public_id
108186,«ՆԵՈ ՄԵՏԱԼՍ» ՍՊԸ,«ՆԵՈ ՄԵՏԱԼՍ»,neo metals spy,neo metals,positive_same_public_id
133360,Орифов Алишер Маруфович,Орифов Алишер Маруфович,orifov alisher marufovich,orifov alisher marufovich,positive_same_public_id
65760,Иргашев Равшан Рустамович,Иргашев Равшан Рустамович,irgashev ravshan rustamovich,irgashev ravshan rustamovich,positive_same_public_id


In [17]:
display(
    pairs_full[pairs_full["pair_type"] == "hard_negative_similar_name"][
        [
            "party_name_1",
            "party_name_2",
            "name_latin_1",
            "name_latin_2",
            "pair_type",
            "name_similarity",
        ]
    ]
    .head(30)
)

,party_name_1,party_name_2,name_latin_1,name_latin_2,pair_type,name_similarity
1,«ԱՄՓ ՀՈԼԴԻՆԳ» ՍՊԸ,«ԱՄՓ ՀՈԼԴԻՆԳՍ» ՓԲԸ,amp holding spy,amp holdings pby,hard_negative_similar_name,90.322581
2,«ԱՄՓ ՀՈԼԴԻՆԳ» ՍՊԸ,«ԱՄՓ ՀՈԼԴԻՆԳՍ» ՓԲԸ,amp holding spy,amp holdings pby,hard_negative_similar_name,90.322581
3,«ԱՄՓ ՀՈԼԴԻՆԳՍ» ՓԲԸ,«ԱՄՓ ՀՈԼԴԻՆԳ» ՍՊԸ,amp holdings pby,amp holding spy,hard_negative_similar_name,90.322581
4,«ԱՄՓ ՀՈԼԴԻՆԳ» ՍՊԸ,«ԱՄՓ ՀՈԼԴԻՆԳՍ» ՓԲԸ,amp holding spy,amp holdings pby,hard_negative_similar_name,90.322581
6,«ԱՄՓ ՀՈԼԴԻՆԳ»,«ԱՄՓ ՀՈԼԴԻՆԳՍ» ՓԲԸ,amp holding,amp holdings pby,hard_negative_similar_name,81.481481
12,«ԱՄՓ ՀՈԼԴԻՆԳ» ՍՊԸ,«ԱՄՓ ՀՈԼԴԻՆԳՍ» ՓԲԸ,amp holding spy,amp holdings pby,hard_negative_similar_name,90.322581
14,«ԱՄՓ ՀՈԼԴԻՆԳ»,«ԱՄՓ ՀՈԼԴԻՆԳՍ» ՓԲԸ,amp holding,amp holdings pby,hard_negative_similar_name,81.481481
16,«ԱՄՓ ՀՈԼԴԻՆԳՍ» ՓԲԸ,«ԱՄՓ ՀՈԼԴԻՆԳ» ՍՊԸ,amp holdings pby,amp holding spy,hard_negative_similar_name,90.322581
22,«ԱՄՓ ՀՈԼԴԻՆԳՍ» ՓԲԸ,«ԱՄՓ ՀՈԼԴԻՆԳ» ՍՊԸ,amp holdings pby,amp holding spy,hard_negative_similar_name,90.322581
23,«ԱՄՓ ՀՈԼԴԻՆԳ»,«ԱՄՓ ՀՈԼԴԻՆԳՍ» ՓԲԸ,amp holding,amp holdings pby,hard_negative_similar_name,81.481481


In [18]:
save_path = "../data/processed/labeled_pairs.parquet"

pairs_full.to_parquet(save_path, index=False)

print(f"Saved to: {save_path}")
print(f"Rows: {len(pairs_full):,}")

Saved to: ../data/processed/labeled_pairs.parquet
Rows: 151,014
